In [1]:
from fixture.factory.feature.closes import factory_closes_n4

closes_n4 = factory_closes_n4()
closes_n4

Date,now,lag_1,lag_2,lag_3,lag_4
datetime[μs],f64,f64,f64,f64,f64
2000-01-06 00:00:00,-35.689539,4.94927,62.57159,33.932168,23.019577
2000-01-07 00:00:00,92.922772,-35.689539,4.94927,62.57159,33.932168
2000-01-08 00:00:00,16.71337,92.922772,-35.689539,4.94927,62.57159
2000-01-09 00:00:00,-102.686526,16.71337,92.922772,-35.689539,4.94927
2000-01-10 00:00:00,62.330155,-102.686526,16.71337,92.922772,-35.689539
…,…,…,…,…,…
2000-04-05 00:00:00,34.229184,155.968606,-55.381002,55.381002,67.05413
2000-04-06 00:00:00,-14.787433,34.229184,155.968606,-55.381002,55.381002
2000-04-07 00:00:00,91.148263,-14.787433,34.229184,155.968606,-55.381002


In [2]:
import lightgbm as lgb
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 特徴量とターゲットの分割（to_pandas()で変換）
X = closes_n4[['lag_1', 'lag_2', 'lag_3', 'lag_4']].to_pandas()
y = closes_n4['now'].to_numpy()  # numpy配列に変換

# 時系列を考慮したデータ分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False, random_state=42
)

# LightGBM用データセット作成（labelをnumpy配列で指定）
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# 回帰用パラメータ設定
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

# モデル訓練
model = lgb.train(
    params,
    train_data,
    valid_sets=[test_data],
    num_boost_round=500,
    callbacks=[lgb.early_stopping(20)]
)

# 予測と評価
y_pred = np.asarray(model.predict(X_test))  # numpy配列に明示的に変換
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Test RMSE: {rmse:.4f}')

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[180]	valid_0's rmse: 66.0734
Test RMSE: 66.0734
